In [2]:
%load_ext autoreload
%autoreload 2

import os
import re
import pandas as pd
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Import your scraper function from the .py file
from scraper_utils import run_scraper

# Define parameters dynamically at runtime
path = os.getenv("PORTAL")
silver = os.getenv("SILVERNEW")

states_input = {
    "5": "UTTARAKHAND", 
    "3": "PUNJAB",
    "2": "HIMACHAL PRADESH", 
    "1": "JAMMU AND KASHMIR", 
    "17": "MEGHALAYA",
    "6": "HARYANA", 
    "16": "TRIPURA",
    "30": "GOA", 
    "34": "PUDUCHERRY",
    "38": "THE DADRA AND NAGAR HAVELI AND DAMAN AND DIU",
    "12": "ARUNACHAL PRADESH", 
    "35": "ANDAMAN AND NICOBAR ISLANDS", 
    "37": "LADAKH",
    "7": "DELHI", 
    "14": "MANIPUR",
    "4": "CHANDIGARH", 
    "15": "MIZORAM", 
    "13": "NAGALAND",
    "31": "LAKSHADWEEP", 
    "11": "SIKKIM"

}

# --- DYNAMIC ACTIVITIES_INPUT GENERATION & STATE ITERATION START ---
stact_path = os.getenv("STACT")
if not stact_path:
    raise ValueError("⚠️ STACT environment variable is not set.")

if not stact_path.lower().endswith((".xlsx", ".xls")):
    stact_path += ".xlsx"

if not os.path.exists(stact_path):
    raise FileNotFoundError(f"⚠️ STACT file not found at: {stact_path}")

# Read the STACT Excel file once
stact_df = pd.read_excel(stact_path)

# Standardize Location values in Excel for comparison
stact_df["_location_clean"] = (
    stact_df["Location"].astype(str).str.strip().str.upper()
)

# Regex to extract key and value from strings like: "15": "4(c) Asbestos milling...",
mapping_pattern = re.compile(r'^\s*"(.*?)":\s*"(.*?)"\s*,\s*$')

# Iterate over each state individually to optimize execution per state
for state_code, state_name in states_input.items():
    current_state_input = {state_code: state_name}
    target_state_clean = state_name.strip().upper()
    
    print(f"\n==================================================")
    print(f"🔄 Processing State: {state_name} (Code: {state_code})")
    print(f"==================================================")

    # Filter rows for the current state only
    matching_rows = stact_df[
        stact_df["_location_clean"] == target_state_clean
    ]

    # DEFENSIVE CHECK 1: Identify if State is Missing in STACT Excel
    if matching_rows.empty:
        print(f"⚠️ WARNING: State '{state_name}' from states_input was NOT found in the STACT Excel file. Skipping...")
        continue

    activities_input = {}
    unique_mappings = matching_rows["Activity Mapping"].dropna().unique()

    for entry in unique_mappings:
        entry_str = str(entry).strip()
        match = mapping_pattern.match(entry_str)
        if match:
            act_id, act_desc = match.groups()
            
            clean_act_id = act_id.strip()
            clean_act_desc = re.sub(r"\s+", " ", act_desc).strip()

            if clean_act_id not in activities_input:
                activities_input[clean_act_id] = clean_act_desc

    # DEFENSIVE CHECK 2: Ensure At Least One Activity Was Found
    if not activities_input:
        print(f"⚠️ WARNING: No valid activities found in STACT Excel for state '{state_name}'. Skipping...")
        continue

    print(f"✅ Loaded {len(activities_input)} unique activity/activities for '{state_name}':")
    for k, v in activities_input.items():
        print(f'   "{k}": "{v}"')

    # Run execution specifically for this state and its corresponding activities
    run_scraper(
        portal_url=path,
        output_base_dir=silver,
        states=current_state_input,
        activities=activities_input,
        year="2026",
        headless=False  # Set to True if you don't want the browser window to open
    )
# --- DYNAMIC ACTIVITIES_INPUT GENERATION & STATE ITERATION END ---

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload

🔄 Processing State: UTTARAKHAND (Code: 5)
✅ Loaded 10 unique activity/activities for 'UTTARAKHAND':
   "11": "3(b) Cement plants"
   "25": "5(g) Distilleries"
   "75": "5(ga) Grain based distilleries"
   "10": "3(a) Metallurgical Industries (ferrous and non ferrous)"
   "1": "1(a) Mining of minerals"
   "27": "5(i) Pulp & Paper Industry"
   "24": "5(f) Synthetic organic chemicals industry"
   "14": "4(b)(ii) Coaltar processing units"
   "28": "5(j) Sugar Industry"
   "6": "1(d) Thermal Power Plants"

🌍 Processing State: UTTARAKHAND (Value: 5)...
  └── ⚙️ Searching Activity ID: 11 (3(b) Cement plants...)
  ℹ️ No results found for State: 5 + Activity: 11. Proceeding...
  └── ⚙️ Searching Activity ID: 25 (5(g) Distilleries...)
  ℹ️ No results found for State: 5 + Activity: 25. Proceeding...
  └── ⚙️ Searching Activity ID: 75 (5(ga) Grain based distilleries...)
  ℹ️ No results found for State: 5 + Acti

KeyboardInterrupt: 